# CMO aircraft motion in 3D

This notebook parses the `PY_CONTACT_LOG` records emitted by `event_export_lua_02.lua` and plots position, altitude, heading, and speed. The grey line is the trajectory, point colour is speed in knots, and red arrows show heading. CMO contact altitude is treated as metres and speed as knots.


In [ ]:
from pathlib import Path
import sys

# Support launches from either the repository root or notebooks directory.
repo_root = Path.cwd() if (Path.cwd() / 'combat_id_calibration').exists() else Path.cwd().parent
sys.path.insert(0, str(repo_root))

from combat_id_calibration.aircraft_motion import (
    AircraftSample, load_contact_log, plot_aircraft_motion
)


## Load an exported CMO log

Set `log_path` to the Lua history file containing the export. Because the Lua script produces one row per emission, the loader removes duplicate kinematic samples by default. `event_export_lua_02.lua` does not export a contact GUID, so filter/group records as appropriate when a sensor sees multiple contacts of the same type.


In [ ]:
log_path = repo_root / 'LuaHistory_2026-06-23.txt'
samples = load_contact_log(log_path) if log_path.exists() else []
# The current Lua schema lacks a contact GUID. Select one coarse track.
if samples:
    track_key = (samples[0].sensor_aircraft, samples[0].target_type)
    samples = [s for s in samples if (s.sensor_aircraft, s.target_type) == track_key]
len(samples)


## Self-contained demonstration

The fallback path below makes the notebook demonstrable even when a CMO history export is not present or contains no `PY_CONTACT_LOG` rows.


In [ ]:
if not samples:
    samples = [
        AircraftSample(str(i), 'Demo sensor', 50.00 + i * 0.025,
                       -1.00 + i * 0.035, (35 + i * 12) % 360,
                       2500 + i * 500, 280 + i * 18, 'Aircraft')
        for i in range(12)
    ]

samples[:3]


In [ ]:
fig, ax, speed_points = plot_aircraft_motion(
    samples, arrow_stride=max(1, len(samples) // 12), arrow_length_km=1.5
)
fig.tight_layout()


## Select one track when a log contains several contacts

Until the Lua export includes a contact GUID, the available grouping fields are the observing aircraft and target type. The example below shows this coarse filtering.


In [ ]:
sensor_names = sorted({sample.sensor_aircraft for sample in samples})
selected = [s for s in samples if s.sensor_aircraft == sensor_names[0]]
print(f'{sensor_names[0]}: {len(selected)} samples')
